# RhymeLM — Evaluation & Visualization

Load a trained checkpoint and analyze what the model has learned:
- Character embedding space (do vowels, consonants, and punctuation cluster?)
- Generation quality across temperatures and sampling strategies
- Evaluation metrics: perplexity, distinct-n, rhyme density, verse structure

In [ ]:
import torch
import numpy as np

from rhymelm.data.tokenizer import CharTokenizer
from rhymelm.models import RhymeLM
from rhymelm.generation.sampler import generate_verse
from rhymelm.evaluation.metrics import evaluate_verse, perplexity
from rhymelm.visualization.embedding_viz import plot_embedding_space
from rhymelm.utils import get_device

## 1. Load Checkpoint

In [ ]:
CHECKPOINT_PATH = "../checkpoints/rhymelm_lstm_final.pt"

device = get_device()
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

tokenizer = CharTokenizer(checkpoint["tokenizer_chars"])
model_cfg = checkpoint["config"]["model"]

model = RhymeLM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=model_cfg["embed_dim"],
    hidden_dim=model_cfg["hidden_dim"],
    num_layers=model_cfg["num_layers"],
    dropout=model_cfg["dropout"],
).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

print(f"Loaded step {checkpoint['step']}")

## 2. Character Embedding Space

A well-trained model should cluster characters by category:
vowels together, consonants together, digits together, punctuation together.
This is a direct visualization of what the embedding layer has learned.

In [ ]:
weights = model.embed.weight.detach().cpu().numpy()
chars = [tokenizer.itos[i] for i in range(tokenizer.vocab_size)]

plot_embedding_space(weights, chars, method="tsne")

## 3. Sampling Strategy Comparison

Compare how different sampling parameters affect output quality.

In [ ]:
strategies = [
    {"name": "Greedy-ish (temp=0.5)", "temperature": 0.5},
    {"name": "Balanced (temp=0.8)", "temperature": 0.8},
    {"name": "Creative (temp=1.0)", "temperature": 1.0},
    {"name": "Top-k=50", "temperature": 0.8, "top_k": 50},
    {"name": "Nucleus p=0.95", "temperature": 0.8, "top_p": 0.95},
    {"name": "Nucleus + rep penalty", "temperature": 0.8, "top_p": 0.95, "repetition_penalty": 1.2},
]

for s in strategies:
    name = s.pop("name")
    print(f"\n{'=' * 50}")
    print(f"Strategy: {name}")
    print(f"{'=' * 50}")
    verse = generate_verse(model, tokenizer, device, prompt="I ", num_bars=8, **s)
    print(verse)
    metrics = evaluate_verse(verse)
    print(f"\n  distinct-1: {metrics['distinct_1']:.3f}  distinct-2: {metrics['distinct_2']:.3f}  "
          f"rhyme: {metrics['rhyme_density']:.3f}  unique: {metrics['unique_line_ratio']:.3f}")

## 4. Batch Evaluation

Generate multiple verses and compute aggregate metrics to get a stable picture of model quality.

In [ ]:
num_samples = 20
all_metrics = []

for i in range(num_samples):
    verse = generate_verse(model, tokenizer, device, temperature=0.8, top_p=0.95, num_bars=16)
    all_metrics.append(evaluate_verse(verse))

# Aggregate
print(f"Averaged over {num_samples} verses:")
for key in all_metrics[0]:
    values = [m[key] for m in all_metrics]
    print(f"  {key}: {np.mean(values):.4f} +/- {np.std(values):.4f}")